In [1]:
# Install missing dependencies in Colab runtime before running the pipeline cells
import importlib.util
import subprocess
import sys

REQUIRED = [
    ("segment_anything", "segment-anything"),
    ("cv2", "opencv-python"),
    ("diffusers", "diffusers"),
    ("transformers", "transformers"),
    ("accelerate", "accelerate"),
    ("fastapi", "fastapi"),
    ("uvicorn", "uvicorn"),
    ("pyngrok", "pyngrok"),
    ("nest_asyncio", "nest_asyncio"),
]

missing_pkgs = [pip_name for mod_name, pip_name in REQUIRED if importlib.util.find_spec(mod_name) is None]
if missing_pkgs:
    print("Installing missing packages:", ", ".join(missing_pkgs))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_pkgs])
else:
    print("All required packages already installed.")

# Some runtimes may still miss segment-anything from PyPI mirror; use GitHub fallback.
if importlib.util.find_spec("segment_anything") is None:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "git+https://github.com/facebookresearch/segment-anything.git",
    ])
    print("Installed segment-anything from GitHub fallback.")
else:
    print("segment_anything import is available.")

All required packages already installed.
segment_anything import is available.


In [2]:
import os
import gc
import json
import torch
import numpy as np
import cv2
import urllib.request
from PIL import Image
from diffusers import (
    StableDiffusionPipeline,
    StableDiffusionControlNetPipeline,
    ControlNetModel,
    StableDiffusionXLInpaintPipeline,
    StableDiffusionInpaintPipeline,
 )
from segment_anything import sam_model_registry, SamAutomaticMaskGenerator
from transformers import CLIPModel, CLIPProcessor

# --- 1. CONFIGURATION & PAYLOAD ---
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32

payload = json.loads(os.environ.get("PIPELINE_PAYLOAD", "{}"))
PROMPT = payload.get("prompt", "a forest at sunset")
NEG_PROMPT = payload.get("negative_prompt", "")
INPAINT_PROMPT = payload.get("inpaint_prompt", PROMPT)
LOW_VRAM = bool(payload.get("low_vram", True))
ENABLE_SDXL = bool(payload.get("enable_sdxl", False))

def _cleanup(*objs):
    for obj in objs:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

def _optimize_pipe(pipe):
    if DEVICE == "cuda":
        try:
            pipe.enable_attention_slicing()
        except Exception:
            pass
        if LOW_VRAM:
            try:
                pipe.enable_model_cpu_offload()
                return pipe
            except Exception:
                pass
    return pipe.to(DEVICE)

def get_canny_image(image):
    """Utility to generate Canny edge maps for ControlNet."""
    gray = np.array(image.convert("L"))
    edges = cv2.Canny(gray, 100, 200)
    return Image.fromarray(edges).convert("RGB")

# --- 2. BASE IMAGE GENERATION (SD 1.5) ---
def generate_base_image():
    pipe = StableDiffusionPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        torch_dtype=DTYPE,
        safety_checker=None
    )
    pipe = _optimize_pipe(pipe)

    with torch.inference_mode():
        image = pipe(
            PROMPT,
            negative_prompt=NEG_PROMPT,
            num_inference_steps=22 if LOW_VRAM else 30,
            guidance_scale=7.0,
        ).images[0]

    image.save("base.png")
    _cleanup(pipe)
    return image

# --- 3. SEGMENTATION (SAM) ---
def segment_image(image):
    sam_ckpt = "sam_vit_b.pth"
    if not os.path.exists(sam_ckpt):
        url = "https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth"
        urllib.request.urlretrieve(url, sam_ckpt)

    sam = sam_model_registry["vit_b"](checkpoint=sam_ckpt).to(DEVICE)
    mask_gen = SamAutomaticMaskGenerator(
        sam,
        points_per_side=16 if LOW_VRAM else 32,
        pred_iou_thresh=0.88,
    )

    masks = mask_gen.generate(np.array(image))
    masks = sorted(masks, key=lambda x: x["area"], reverse=True)[:5 if LOW_VRAM else 10]
    _cleanup(sam)
    return masks

# --- 4. CLASSIFICATION (CLIP) ---
def label_segments(image, masks):
    labels = ["sky", "person", "tree", "building", "water", "road", "car", "animal", "grass"]
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(DEVICE)
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    results = []
    for m in masks:
        x, y, w, h = [int(v) for v in m["bbox"]]
        crop = image.crop((x, y, x + w, y + h))

        inputs = processor(text=labels, images=crop, return_tensors="pt", padding=True).to(DEVICE)
        with torch.inference_mode():
            logits = model(**inputs).logits_per_image
            probs = logits.softmax(dim=1)[0]

        idx = probs.argmax().item()
        results.append({
            "label": labels[idx],
            "confidence": round(float(probs[idx]), 3),
            "bbox": m["bbox"]
        })

    _cleanup(model)
    return results

# --- 5. EDITING & INPAINTING ---
def apply_edits(base_img, masks):
    # Step A: ControlNet Canny Edit (SD 1.5)
    cn_model = ControlNetModel.from_pretrained(
        "lllyasviel/sd-controlnet-canny",
        torch_dtype=DTYPE,
    ).to(DEVICE)
    cn_pipe = StableDiffusionControlNetPipeline.from_pretrained(
        "runwayml/stable-diffusion-v1-5",
        controlnet=cn_model,
        torch_dtype=DTYPE,
    )
    cn_pipe = _optimize_pipe(cn_pipe)

    canny_base = get_canny_image(base_img)
    with torch.inference_mode():
        edited_img = cn_pipe(
            PROMPT,
            image=canny_base,
            controlnet_conditioning_scale=0.8,
            num_inference_steps=20 if LOW_VRAM else 28,
            guidance_scale=7.0,
        ).images[0]
    edited_img.save("edited.png")

    _cleanup(cn_model, cn_pipe)

    # Step B: Optional inpainting pass
    if not masks or not ENABLE_SDXL:
        return edited_img

    mask_pil = Image.fromarray((np.array(masks[0]["segmentation"]) * 255).astype("uint8"))

    try:
        if LOW_VRAM:
            inpaint_pipe = StableDiffusionInpaintPipeline.from_pretrained(
                "runwayml/stable-diffusion-inpainting",
                torch_dtype=DTYPE,
            )
            inpaint_pipe = _optimize_pipe(inpaint_pipe)

            with torch.inference_mode():
                final_img = inpaint_pipe(
                    prompt=INPAINT_PROMPT,
                    negative_prompt=NEG_PROMPT,
                    image=edited_img,
                    mask_image=mask_pil,
                    strength=0.95,
                    num_inference_steps=20,
                ).images[0]
            _cleanup(inpaint_pipe)
        else:
            xl_pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
                "diffusers/stable-diffusion-xl-1.0-inpainting-0.1",
                torch_dtype=DTYPE,
            )
            xl_pipe = _optimize_pipe(xl_pipe)

            with torch.inference_mode():
                final_img = xl_pipe(
                    prompt=INPAINT_PROMPT,
                    image=edited_img,
                    mask_image=mask_pil,
                    strength=0.99,
                    num_inference_steps=24,
                ).images[0]
            _cleanup(xl_pipe)

        final_img.save("final.png")
        return final_img
    except RuntimeError as exc:
        print("Inpainting step failed, falling back to edited image:", exc)
        _cleanup()
        return edited_img

def run_pipeline_once(active_payload):
    """Execute full pipeline and write result.json for API response assembly."""
    global PROMPT, NEG_PROMPT, INPAINT_PROMPT, LOW_VRAM, ENABLE_SDXL
    PROMPT = active_payload.get("prompt", PROMPT)
    NEG_PROMPT = active_payload.get("negative_prompt", NEG_PROMPT)
    INPAINT_PROMPT = active_payload.get("inpaint_prompt", PROMPT)
    LOW_VRAM = bool(active_payload.get("low_vram", True))
    ENABLE_SDXL = bool(active_payload.get("enable_sdxl", False))

    print("--- Starting Pipeline ---")
    img_base = generate_base_image()
    segments = segment_image(img_base)
    labels = label_segments(img_base, segments)
    img_final = apply_edits(img_base, segments)

    with open("result.json", "w") as f:
        json.dump(
            {
                "segments": labels,
                "outputs": ["base.png", "edited.png", "final.png"],
                "execution_mode": "colab",
                "gpu_device": DEVICE,
                "low_vram": LOW_VRAM,
                "enable_sdxl": ENABLE_SDXL,
            },
            f,
        )
    print("--- Pipeline Complete ---")
    return img_base, labels, img_final

print("Pipeline functions loaded. Use API bridge cell to run generation.")

Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


Pipeline functions loaded. Use API bridge cell to run generation.


In [3]:
# Required for DNS-safe tunnel from many local networks
# Paste your token from https://dashboard.ngrok.com/get-started/your-authtoken
os.environ["NGROK_AUTHTOKEN"] = "3BKfjJQ0Kge9V3fH9x8igX04jok_7KgeryfJGE8WisgzzZvxy"  # e.g. 2abc...
os.environ["ALLOW_CLOUDFLARE_FALLBACK"] = "false"  # keep false on networks that cannot resolve trycloudflare.com

if not os.environ["NGROK_AUTHTOKEN"].strip():
    raise RuntimeError(
        "NGROK_AUTHTOKEN is empty. Paste your token in this cell, then run again. "
        "Do not proceed to the bridge cell until this passes."
    )

masked = os.environ["NGROK_AUTHTOKEN"][:6] + "..."
print("NGROK_AUTHTOKEN loaded:", masked)
print("Now run the next cell to start bridge + ngrok tunnel.")

NGROK_AUTHTOKEN loaded: 3BKfjJ...
Now run the next cell to start bridge + ngrok tunnel.


In [4]:
# Colab API server for backend integration
!pip -q install fastapi uvicorn pyngrok nest_asyncio

import io
import os
import re
import time
import json
import base64
import threading
import subprocess
from typing import Any, Dict

import uvicorn
import nest_asyncio
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from pyngrok import ngrok

nest_asyncio.apply()

class GeneratePayload(BaseModel):
    prompt: str = "a forest at sunset"
    negative_prompt: str = ""
    inpaint_prompt: str | None = None
    job_id: str | None = None
    low_vram: bool = True
    enable_sdxl: bool = False

app = FastAPI(title="AI Image Studio Colab Bridge")

def _img_to_b64(img):
    buf = io.BytesIO()
    img.save(buf, format="PNG")
    return base64.b64encode(buf.getvalue()).decode("utf-8")

@app.get("/health")
def health():
    return {"status": "ok", "device": DEVICE}

@app.post("/generate")
def generate_api(req: GeneratePayload):
    payload: Dict[str, Any] = req.model_dump()
    payload["inpaint_prompt"] = payload.get("inpaint_prompt") or payload["prompt"]
    payload.setdefault("low_vram", True)
    payload.setdefault("enable_sdxl", False)
    os.environ["PIPELINE_PAYLOAD"] = json.dumps(payload)

    try:
        img_base, labels, img_final = run_pipeline_once(payload)

        from PIL import Image as _PILImage
        edited_path = "edited.png"
        if os.path.exists(edited_path):
            img_edited = _PILImage.open(edited_path).convert("RGB")
        else:
            img_edited = img_base

        return {
            "job_id": payload.get("job_id"),
            "base_image": _img_to_b64(img_base),
            "edited_image": _img_to_b64(img_edited),
            "final_image": _img_to_b64(img_final),
            "segments": labels,
            "execution_mode": "colab",
            "gpu_device": DEVICE,
            "low_vram": payload.get("low_vram", True),
            "enable_sdxl": payload.get("enable_sdxl", False),
        }
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc))

def _ensure_cloudflared_binary():
    if subprocess.run(["bash", "-lc", "command -v cloudflared >/dev/null 2>&1"], check=False).returncode == 0:
        return
    subprocess.run(
        [
            "bash",
            "-lc",
            "wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared",
        ],
        check=True,
    )

def _open_cloudflare_tunnel(port: int):
    _ensure_cloudflared_binary()
    log_path = "/tmp/cloudflared.log"
    try:
        os.remove(log_path)
    except FileNotFoundError:
        pass

    subprocess.run(["bash", "-lc", "pkill -f 'cloudflared tunnel --url'"], check=False)

    proc = subprocess.Popen(
        [
            "cloudflared",
            "tunnel",
            "--url",
            f"http://127.0.0.1:{port}",
            "--no-autoupdate",
            "--logfile",
            log_path,
            "--loglevel",
            "info",
        ],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.STDOUT,
        text=True,
    )

    pattern = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
    for _ in range(120):
        time.sleep(1)
        if os.path.exists(log_path):
            content = open(log_path, "r", encoding="utf-8", errors="ignore").read()
            match = pattern.search(content)
            if match:
                return match.group(0).rstrip("/"), proc
        if proc.poll() is not None:
            break

    tail = ""
    if os.path.exists(log_path):
        lines = open(log_path, "r", encoding="utf-8", errors="ignore").read().splitlines()
        tail = "\n".join(lines[-20:])
    proc.terminate()
    raise RuntimeError("Failed to obtain Cloudflare tunnel URL. Cloudflared logs:\n" + (tail or "<no logs>"))

def _open_public_tunnel(port: int):
    # Prefer ngrok because some local networks cannot resolve trycloudflare.com domains.
    ngrok_token = os.getenv("NGROK_AUTHTOKEN") or os.getenv("COLAB_NGROK_AUTHTOKEN")
    allow_cloudflare_fallback = os.getenv("ALLOW_CLOUDFLARE_FALLBACK", "false").lower() == "true"

    if not ngrok_token and not allow_cloudflare_fallback:
        raise RuntimeError(
            "NGROK_AUTHTOKEN is not set. This notebook now prefers ngrok to avoid DNS failures on trycloudflare domains. "
            "Set NGROK_AUTHTOKEN in Colab, then re-run this cell. "
            "If you still want Cloudflare fallback, set ALLOW_CLOUDFLARE_FALLBACK=true."
        )

    if ngrok_token:
        ngrok.set_auth_token(ngrok_token)
        try:
            ngrok.kill()
        except Exception:
            pass
        try:
            url = ngrok.connect(port).public_url.rstrip("/")
            return url, "ngrok", None
        except Exception as ngrok_exc:
            print("ngrok failed:", str(ngrok_exc))
            if not allow_cloudflare_fallback:
                raise RuntimeError("ngrok failed and Cloudflare fallback is disabled.") from ngrok_exc

    print("Using Cloudflare quick tunnel fallback.")
    cf_url, cf_proc = _open_cloudflare_tunnel(port)
    return cf_url, "cloudflared", cf_proc

port = int(os.environ.get("COLAB_BRIDGE_PORT", "8000"))
if "thread" in globals() and getattr(thread, "is_alive", lambda: False)():
    print(f"API server already running on port {port}.")
else:
    thread = threading.Thread(
        target=lambda: uvicorn.run(app, host="0.0.0.0", port=port, log_level="info"),
        daemon=True,
        name="colab-fastapi-server",
    )
    thread.start()

public_base_url, tunnel_provider, cloudflared_proc = _open_public_tunnel(port)
print("Tunnel provider:", tunnel_provider)
print("Public base URL:", public_base_url)
print("Set COLAB_SERVER_URL=", f"{public_base_url}/generate")
print("Set COLAB_HEALTHCHECK_URL=", f"{public_base_url}/health")
if tunnel_provider == "ngrok":
    print("ngrok URL ready. Use these values in your local .env.")

INFO:     Started server process [2307]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


Tunnel provider: ngrok                                                                              
Public base URL: https://spongiest-noneagerly-ursula.ngrok-free.dev
Set COLAB_SERVER_URL= https://spongiest-noneagerly-ursula.ngrok-free.dev/generate
Set COLAB_HEALTHCHECK_URL= https://spongiest-noneagerly-ursula.ngrok-free.dev/health
ngrok URL ready. Use these values in your local .env.
